# C8-embeddings — Session 1: Tokens and Embeddings

*One class session, roughly 80 minutes. Prerequisites: F2-vectors (dot
products, norms, cosine similarity, unit vectors), F3-matrices (matrices
as stacked rows; the matrix product — needed next session), and
F1-scientific-python (arrays, broadcasting, axis sums, seeded
randomness, matplotlib).*

**This session:** how raw text becomes a list of **tokens**
(`gensim.utils.simple_preprocess` and exactly what it keeps and drops);
tokens versus types, and precisely what Python's `set` forgets when you
deduplicate with it; **word embeddings** — each word as a learned dense
vector, with the one governing fact *similar use → nearby vectors*;
loading the pretrained GloVe vectors through `gensim`
(`KeyedVectors`: membership, lookup, dimensionality, and this course's
float64 boundary cast); filtering a token list against the model's
vocabulary; and first geometric measurements on real word vectors with
F2's tools.

Nothing in this unit trains anything: the vectors are a fixed, published
artifact, which is exactly why every number below is reproducible.

Try every checkpoint by hand first, then verify in code.
Answers are collected at the end of this notebook.

In [ ]:
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import numpy as np
from gensim.utils import simple_preprocess

## 1. From Text to Tokens

**Motivation.**
Every model in this course eats numbers — arrays with shapes.
Text arrives as one long string, so any text pipeline must first decide
what the *units* are.
Chopping a string into a list of standardized word-pieces is called
**tokenization**, and each resulting piece is a **token**.

**Definition (tokenization, this course's register).**
We tokenize with `gensim.utils.simple_preprocess`, which takes a string
and returns a list of tokens after applying, in one pass:

- **lowercase** everything (`"The"` and `"the"` become the same token);
- **strip punctuation and digits** — tokens are contiguous runs of
  letters, so `"a.m."` and `"90-meter"` break apart at the dots and
  hyphen, and pure numbers like `"14"` vanish entirely;
- **drop very short and very long pieces** — only pieces between 2 and
  15 characters survive (so the stranded `"a"` and `"m"` from `"a.m."`
  are dropped too).

The exact rules matter less than the habit of *checking them on a
sentence you can eyeball*.
Here is a fresh passage — our running text for this session.

In [ ]:
TEXT = ("The lighthouse keeper climbed the spiral stairs at 5 a.m. to trim the wick "
        "and light the lamp. The lamp blinked twice, and the harbor answered with "
        "two horns. By noon the keeper had logged 14 ships, 3 gulls, and one stubborn seal.")

tokens = simple_preprocess(TEXT)
print(tokens)
print("token count:", len(tokens))

Thirty-nine tokens.
Check the rules against what you see: every capital letter is gone
(`the` leads the list, not `The`); the numerals `5`, `14`, and `3` are
gone; `a.m.` contributed nothing (its splinters `a` and `m` are
single letters, below the 2-character floor); and every comma and period
has disappeared.
What survives is exactly the standardized word material — repeated words
and all.
That repetition is data, and the next section is about not destroying it
by accident.

### Checkpoint 1

1. By hand, apply the three rules to the string
   `"At 9 p.m. the 2 ferries docked."` and write the resulting token
   list. Then verify with `simple_preprocess`.
2. Which of these inputs produce the *same* token list?
   `"The lamp!"`, `"the lamp"`, `"THE LAMP."`, `"the 2 lamps"`.
   Answer before running.
3. In one sentence: why does a tokenizer that drops numerals suit a
   *word*-vector model like this unit's GloVe?

## 2. Tokens, Types, and the Set Trap

**Definitions.**
A **token** is one occurrence in the running text; a **type** is a
distinct word.
Our passage has 39 tokens but far fewer types — `the` alone accounts for
seven tokens.
Counting both, and knowing which one a question asks for, is a standing
exam skill.

`list` methods handle the token side: `len(tokens)` counts occurrences,
`tokens.count("the")` counts one word's occurrences, and
`tokens.index("lamp")` finds a word's *first* position.

In [ ]:
print("tokens          :", len(tokens))
print("count of 'the'  :", tokens.count("the"))
print("count of 'lamp' :", tokens.count("lamp"))
print("count of 'keeper':", tokens.count("keeper"))
print("first 'lamp' at index:", tokens.index("lamp"))

**The type count — and the set trap.**
Python's `set(tokens)` collapses the list to its distinct elements, so
`len(set(tokens))` counts types.
That is the *right* tool for counting and for membership tests.
But a `set` pays for that speed by discarding two things a text pipeline
often still needs:

1. **Order.** A set has no first element.
   Iterating over a set yields its elements in an *arbitrary* order that
   can even change between Python runs — never read "the first words of
   the passage" off a set.
2. **Counts.** A set keeps one copy of everything.
   The fact that `the` appeared seven times and `lamp` twice is simply
   gone.

When you need *deduplication that remembers first-occurrence order*, the
idiom is `list(dict.fromkeys(tokens))`: dictionaries preserve insertion
order, so each token keeps the position of its first appearance.

In [ ]:
n_types = len(set(tokens))
print("types:", n_types)

# membership is what sets are for
print("'harbor' occurs:", "harbor" in set(tokens))

# two orderings, one set — sets cannot remember which came first
print("order forgotten:", set(["tide", "lamp"]) == set(["lamp", "tide"]))

# order-preserving deduplication: first occurrences, in order
uniq_ordered = list(dict.fromkeys(tokens))
print("ordered types:", len(uniq_ordered))
print("first six, in first-appearance order:", uniq_ordered[:6])

# if you ever must display a set, sort it first -- deterministic output
print("alphabetical sample:", sorted(set(tokens))[:6])

The passage has 29 types.
`uniq_ordered` starts `['the', 'lighthouse', 'keeper', ...]` — the
story's actual opening — while the raw set would happily start anywhere.
Never print a bare `set` and rely on what you see: sort it, or use the
`dict.fromkeys` idiom when order carries meaning.

### Checkpoint 2

1. By hand from the printed token list: how many *tokens* and how many
   *types* does the fragment `"the lamp the lamp blinked"` have, and
   what does `list(dict.fromkeys(...))` return for it?
2. A survey pipeline stores respondents' answers as
   `set(["yes", "no", "yes", "no", "no"])`.
   Name the two pieces of information that are now unrecoverable.
3. Why is `sorted(set(tokens))` safe to print in a report while
   `list(set(tokens))` is not — even though both contain exactly the
   same elements?

## 3. Word Embeddings: Learned Dense Vectors

**Motivation.**
Tokens are still strings, and strings have no geometry: you cannot ask
how far `harbor` is from `boat`.
The fix is to assign every vocabulary word a vector.

**Definition (word embedding).**
A **word embedding** assigns each word $t$ in a fixed vocabulary a dense
vector $v_t \in \mathbb{R}^d$ (here $d = 100$).
"Dense" means the entries are ordinary real numbers, almost none of them
zero — the information is spread across all 100 coordinates.

**Where the numbers come from — the facts we take as given.**
The vectors this unit uses are **GloVe** vectors
(`glove-wiki-gigaword-100`), *learned* from co-occurrence statistics of
a six-billion-token corpus (Wikipedia plus a news archive) and published
as a fixed artifact.
Training such vectors is outside this course's scope; using them well is
squarely inside it.
Three stated facts govern everything we do with them:

1. **Similar use → nearby vectors.**
   Words that occur in similar contexts (`boat` and `sailor`, `pepper`
   and `honey`) end up with vectors pointing in similar directions.
   That is the entire reason word geometry is useful.
2. **Individual coordinates mean nothing.**
   Dimension 37 is not "size" or "color"; only *relative geometry* —
   angles and distances between vectors — carries meaning.
3. **The artifact is fixed.**
   Same file, same vectors, every load — which is why this unit can
   assert exact numbers against it.

**What embeddings are not.**
They are not spelling: `lamp` and `lamb` differ by one letter but live
far apart, while `lamp` and `lantern` share little spelling and live
close.
They are not counts, and no human chose the entries.

### Checkpoint 3

1. A classmate says "dimension 12 of GloVe measures how positive a word
   is." Which stated fact does this violate?
2. Predict which pair sits closer in embedding space:
   (`violin`, `cello`) or (`violin`, `garlic`) — and name the stated
   fact your prediction uses.
3. Why does the fixed-artifact fact (3) matter for a course whose
   correctness checks are executable `assert`s?

## 4. Loading GloVe: `gensim` KeyedVectors

**The loader and the cache.**
`gensim.downloader.load` fetches a named artifact once, caches it on
disk, and returns it.
The header cell at the top of this notebook pointed the cache at this
repository's shared location (`reference/cache/gensim`) *before*
importing gensim — that is why the path juggling with `pathlib`
happened up there: every notebook in this repo, whatever directory it
runs from, resolves the repo root (the directory holding
`pyproject.toml`) and reuses one cache instead of re-downloading
130 MB per notebook.

The loaded object is a `KeyedVectors`: a table mapping each vocabulary
word to its vector.
(The first load of a session parses the artifact and takes a little
while; that is normal.)

In [ ]:
import gensim.downloader

kv = gensim.downloader.load("glove-wiki-gigaword-100")
print(type(kv).__name__)
print("vocabulary size:", len(kv.key_to_index))
print("vector dimension:", kv.vector_size)

400,000 words, 100 numbers each.

**Lookup — and the float64 boundary cast.**
Indexing with a word, `kv["harbor"]`, returns that word's vector.
One wrinkle: gensim stores the artifact in **float32**, while this
course's numeric register is **float64** (every anchor and tolerance in
our units assumes it).
The rule, applied *every time vectors leave gensim, in the same cell as
the lookup*: wrap the result in
`np.asarray(..., dtype=np.float64)`.
Downstream arithmetic then lives entirely in float64.

In [ ]:
raw = kv["harbor"]
print("gensim gives:", raw.dtype, raw.shape)

v_harbor = np.asarray(kv["harbor"], dtype=np.float64)   # the boundary cast
print("we keep     :", v_harbor.dtype, v_harbor.shape)
print("first three coordinates:", np.round(v_harbor[:3], 5))

The coordinates themselves (`-0.11293`, `0.20741`, ...) are
uninterpretable, exactly as fact 2 of Section 3 promised — their power
appears only when we *compare* vectors (Section 6).

**Membership.**
Not every string is in the vocabulary.
The membership test is `word in kv.key_to_index` — a plain dictionary
lookup, cheap enough to run over whole token lists.

In [ ]:
for w in ["harbor", "wick", "unlighthouse"]:
    print(f"{w!r:16} in vocabulary: {w in kv.key_to_index}")

### Checkpoint 4

1. What are the shape and dtype of `np.asarray(kv["tide"], dtype=np.float64)`,
   without running it?
2. Why must the float64 cast happen *at the boundary* (in the lookup
   cell) rather than "somewhere later, before the final answer"?
3. Write the one-line expression that checks whether the string
   `"breakwater"` can be embedded by this model.

## 5. Vocabulary Filtering

**Motivation.**
Real token lists contain words the model has never seen —
**out-of-vocabulary (OOV)** tokens: rare names, typos, invented words.
Asking `kv` for one raises a `KeyError`, so every pipeline filters its
tokens against the vocabulary *before* looking anything up.

The standard prelude, start to finish: tokenize, deduplicate *keeping
first-occurrence order* (Section 2's idiom), then keep only in-vocabulary
tokens.
Here it is on a sentence featuring a boat name no corpus has met.

In [ ]:
SENT = "The dinghy Quokkaroo slipped past the breakwater at dusk."

sent_tokens = simple_preprocess(SENT)
print("tokens:", sent_tokens)

dropped = [t for t in sent_tokens if t not in kv.key_to_index]
print("OOV, dropped:", dropped)

candidates = [t for t in dict.fromkeys(sent_tokens) if t in kv.key_to_index]
print("kept (ordered, unique, in-vocabulary):", candidates)

`quokkaroo` — lowercased from the boat's name — is not among the 400,000
words and is dropped; everything else survives, deduplicated
(`the` appears once) and still in reading order.
This exact three-step prelude opens Session 3's retrieval pipeline.

### Checkpoint 5

1. Predict, then verify: what does the filter keep from
   `"The Quokkaroo passed the Quokkaroo buoy."`?
2. Why must the OOV filter run *before* any `kv[...]` lookup rather
   than inside a `try/except` around the whole pipeline? Give one
   concrete advantage.
3. In the filter's list comprehension, what changes if you swap
   `dict.fromkeys(sent_tokens)` for `set(sent_tokens)` — and why is
   that a bug here?

## 6. First Measurements: Norms, Dots, Cosines

**Motivation.**
With words as float64 vectors, F2's toolkit applies verbatim.
Recall the three instruments, in this unit's broadcasting register (no
`np.linalg` — the exam's implementation tasks ban it, so we never form
the habit):

- **norm**: $\lVert v \rVert = \sqrt{\sum_k v_k^2}$
  — in code `np.sqrt((v * v).sum())`;
- **dot product**: $u \cdot v = \sum_k u_k v_k$
  — in code `(u * v).sum()`;
- **cosine similarity**:
  $$\cos(u, v) \;=\; \frac{u \cdot v}{\lVert u \rVert\, \lVert v \rVert}
  \;\in\; [-1, 1],$$
  the dot product with both lengths divided out — pure *direction*
  agreement: $1$ means same direction, $0$ unrelated, $-1$ opposite.

Cosine is the standard yardstick for embeddings precisely because
stated fact 1 speaks of nearby *directions*; a word's raw vector length
tracks corpus quirks we do not want to reward.

In [ ]:
def cosine(u, v):
    """Cosine similarity of two 1-D float64 vectors (no np.linalg, no loops)."""
    nu = np.sqrt((u * u).sum())
    nv = np.sqrt((v * v).sum())
    return (u * v).sum() / (nu * nv)


print("||harbor|| =", round(float(np.sqrt((v_harbor * v_harbor).sum())), 4))

for a, b in [("harbor", "boat"), ("boat", "sailor"), ("harbor", "pepper")]:
    u = np.asarray(kv[a], dtype=np.float64)
    v = np.asarray(kv[b], dtype=np.float64)
    print(f"cos({a}, {b}) = {cosine(u, v):.4f}")

Read the scale like a thermometer: `harbor`–`boat` at $0.60$ is strongly
related, `boat`–`sailor` at $0.47$ clearly related, `harbor`–`pepper`
at $0.09$ essentially unrelated.
Two calibration points to memorize: identical words score exactly $1$;
typical *unrelated* word pairs score near $0$ (slightly above, on
average) rather than near $-1$ — strong negative cosines are rare among
real words.

Session 2 scales these pairwise calls into one matrix expression for
*all* pairs at once.

### Checkpoint 6

1. By hand: $u = (3, 4)$, $v = (8, 6)$ — compute $\cos(u, v)$ exactly.
2. Without computing: what is `cosine(kv_vec("lamp"), kv_vec("lamp"))`
   for any word, and why does the answer not depend on the word?
3. `cos(u, v) = 0.999` yet `np.sqrt(((u - v)**2).sum())` is large.
   How can both hold at once? (Think direction versus length.)

## 7. Worked Exam-Style Example: Tokens and Types MC

Round 1 opens its text problems with exactly this register: a short
string, a tokenizer, and a count that punishes sloppy rule-reading.
Worked in full, the way you should work it at the desk.

---

**Problem.**
Let `s = "The 3 ferries left Dock 9; the dock kept none."` and
`toks = simple_preprocess(s)`.
What is `(len(toks), len(set(toks)))`?

A. `(8, 6)`  B. `(8, 7)`  C. `(9, 7)`  D. `(10, 8)`  E. `(10, 10)`

*Reasoning is not required (but we reason anyway).*

---

**Step 1 — apply the tokenizer rules by hand.**
Lowercase everything; digits `3` and `9` die; punctuation splits and
vanishes; every surviving piece has 2–15 letters:

`['the', 'ferries', 'left', 'dock', 'the', 'dock', 'kept', 'none']`

Count: **8 tokens**. That eliminates C, D, E immediately.

**Step 2 — count types.**
Distinct words: `the`, `ferries`, `left`, `dock`, `kept`, `none` —
**6 types** (`the` and `dock` each appear twice; note that `Dock` and
`dock` merged *because* of lowercasing — a favorite trap).

**Step 3 — decide and verify.**
`(8, 6)` — answer **A**.
The wrong answers are manufactured from real mistakes: B forgets that
lowercasing merges `Dock` into `dock`; C keeps one numeral; D keeps
both numerals; E confuses tokens with types on top of that.

In [ ]:
s = "The 3 ferries left Dock 9; the dock kept none."
toks = simple_preprocess(s)
print(toks)
print((len(toks), len(set(toks))))   # (8, 6) -> answer A

### Checkpoint 7

1. Rework the example with
   `s = "Nine lamps; nine wicks, 9 flames."` — compute
   `(len(toks), len(set(toks)))` by hand, then verify.
2. Build a five-option MC of your own around a one-line string whose
   token and type counts differ by exactly 2, and mark the correct
   option.

## 8. Common Pitfalls I

**Pitfall 1 — case and OOV `KeyError`s.**
The vocabulary is lowercase (the tokenizer guarantees it), so raw-string
lookups with capitals fail even for common words.
Symptom: `KeyError: "Key 'Harbor' not present"`.
Habit: *only ever look up tokens that came out of the
tokenize-and-filter prelude*, never raw strings from the source text.

In [ ]:
try:
    kv["Harbor"]
except KeyError as e:
    print("KeyError:", e)

print("after the prelude:", simple_preprocess("Harbor!"),
      "->", "harbor" in kv.key_to_index)

**Pitfall 2 — trusting a set's order.**
`list(set(tokens))[0]` is not "the first token"; it is whatever the hash
table happened to put first, and it can differ between two runs of the
same script.
If an answer depends on order, the set was the wrong container —
reach for `dict.fromkeys`.
The symptom is the nastiest kind: code that *passes today* and fails on
the grader's machine.

**Pitfall 3 — the float32 leak.**
Skip the boundary cast and every downstream number quietly inherits
float32's ~7 significant digits.
The demonstration: the same norm computed both ways.

In [ ]:
raw32 = kv["harbor"]                                   # float32, no cast
v64 = np.asarray(kv["harbor"], dtype=np.float64)       # the register

norm32 = np.sqrt((raw32 * raw32).sum())
norm64 = np.sqrt((v64 * v64).sum())
print("float32 norm:", repr(float(norm32)))
print("float64 norm:", repr(float(norm64)))
print("difference  :", abs(float(norm32) - float(norm64)))

The two norms disagree in the seventh digit.
Against this unit's float64 anchors (checked at `1e-6` and tighter),
uncast arithmetic fails asserts *intermittently* — the worst kind of
wrong.
The cast costs one wrapper at the boundary; put it in the lookup cell,
every time.

**Pitfall 4 — tokens when types were asked (and vice versa).**
The words "how many words" are ambiguous in English and never ambiguous
in an exam: *occurrences* means `len(tokens)`, *distinct* means
`len(set(tokens))`.
Underline which one the problem asks for before computing — Section 7's
option E exists purely to catch the unhurried reader who did not.

### Checkpoint 8

1. A teammate's script prints `list(set(toks))[:3]` as "the passage's
   first three words" and passed all their local runs. Which pitfall,
   and why might it *still* pass on their machine every time?
2. A teammate computes `harbor`'s norm as `5.585229397`; yours is
   `5.585229575`. Neither of you made an arithmetic slip. Which
   pitfall explains the gap, which of the two numbers is this
   course's canonical value, and what one-line change brings the
   teammate's to match yours to twelve digits?
3. Write the two expressions for "how many words are in `toks`" — one
   per reading of the question.

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. `['at', 'the', 'ferries', 'docked']` — `At`→`at`, `9` and `2` are
   digits (dropped), `p.m.` shatters into single letters (dropped),
   `docked` keeps its letters and loses the period.
2. The first three are identical: all yield `['the', 'lamp']`.
   The fourth yields `['the', 'lamps']` — `lamps` is a different token,
   and `2` is dropped.
3. GloVe's vocabulary is made of *words*; numerals are unbounded in
   variety and mostly absent from the vocabulary, so keeping them would
   manufacture OOV tokens with no vector to look up.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. 5 tokens, 3 types; `list(dict.fromkeys(...))` returns
   `['the', 'lamp', 'blinked']`.
2. The tally (three "no", two "yes") and the order in which the answers
   arrived — the set keeps only `{"yes", "no"}`.
3. `sorted(...)` imposes one deterministic order (alphabetical) on the
   same elements; a bare `list(set(...))` inherits the hash table's
   arbitrary, run-dependent order.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Fact 2: individual coordinates carry no assigned meaning — only the
   geometry between whole vectors does.
2. (`violin`, `cello`): fact 1 — the two words occur in similar
   contexts (orchestras, strings, concerts), so their vectors point in
   nearby directions; `garlic` shares almost no contexts with `violin`.
3. Because asserts check exact numbers: only a fixed artifact makes
   `assert np.isclose(...)` reproducible on every machine and every
   run. A re-trained embedding would shift every anchor.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. Shape `(100,)`, dtype `float64` — one coordinate per embedding
   dimension, cast by the wrapper.
2. Because *every* intermediate result computed before the cast is
   already float32-precision; casting later launders the dtype label
   but cannot restore the lost digits. The boundary is the only place
   the cast is airtight.
3. `"breakwater" in kv.key_to_index`

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Tokens: `['the', 'quokkaroo', 'passed', 'the', 'quokkaroo', 'buoy']`;
   the filter keeps `['the', 'passed', 'buoy']` — both `quokkaroo`
   occurrences are OOV, and `the` is deduplicated to its first
   position.
2. The filter documents *which* tokens were dropped (you can print
   `dropped` and reason about it); a blanket `try/except` silently
   loses that information and can also mask unrelated `KeyError`s from
   genuine bugs.
3. The kept tokens come out in the set's arbitrary order instead of
   first-occurrence order — the list's *contents* match, but any
   downstream step that reads order (display, ranking rows of a
   matrix) is now nondeterministic.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $u \cdot v = 24 + 24 = 48$; $\lVert u \rVert = 5$,
   $\lVert v \rVert = 10$; $\cos = 48/50 = 24/25 = 0.96$.
2. Exactly $1$: $\cos(v, v) = (v \cdot v) / (\lVert v \rVert\,\lVert
   v \rVert) = \lVert v \rVert^2 / \lVert v \rVert^2$ — the norms
   cancel for *any* nonzero vector, so the word does not matter.
3. $v = 10u$ (say): the direction gap is tiny (cosine near 1) while
   the *length* gap makes the Euclidean distance large. Cosine ignores
   length by construction.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. `['nine', 'lamps', 'nine', 'wicks', 'flames']` — the numeral `9`
   dies but the *word* `nine` is letters and survives, twice.
   `(5, 4)`.
2. (Open-ended.) Any string where one word repeats once and a numeral
   or single letter pads the raw count works; check your five options
   include the "kept the numeral" and "counted types" traps.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. Pitfall 2. Within one Python *process* a given set often iterates
   in a stable order, and small string sets can even look sorted by
   accident — the failure only appears under a different hash seed,
   i.e. on another machine or another day.
2. Pitfall 3 (the float32 leak): the two values agree to about seven
   significant digits — float32's precision, exactly the gap the
   Pitfall-3 demo printed ($\approx 1.8 \times 10^{-7}$). The float64
   value `5.585229575` is canonical (the course register). Fix:
   `v = np.asarray(kv["harbor"], dtype=np.float64)` at the lookup.
3. Occurrences: `len(toks)`. Distinct words: `len(set(toks))`.

</details>